<a href="https://colab.research.google.com/github/mbimira/My1strepo/blob/main/A_B_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simulated Email Subject-Line Experiment

> **Synthetic demonstration only:** this notebook creates fictional recipients and randomly generated outcomes. It does not send emails or analyze a real campaign. Rates and p-values here are not evidence that either subject line performs better.

This small example demonstrates random assignment, funnel metrics, and a test for one primary metric.

## Simulation assumptions

- 100 fictional recipients are randomly assigned to two groups.
- Both variants use the same assumed open probability (55%), so the simulation has no treatment effect built in.
- Clicks are generated only for recipients who opened; conversions are generated only for those who clicked.
- **Primary metric:** open rate. Click and conversion rates are descriptive secondary metrics.

The sample size and probabilities are chosen for demonstration, not from a power analysis or business requirement.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact

rng = np.random.default_rng(42)
n_users = 100

variant = np.repeat(["A", "B"], n_users // 2)
rng.shuffle(variant)
opened = rng.binomial(1, 0.55, size=n_users)
clicked = opened * rng.binomial(1, 0.20, size=n_users)
converted = clicked * rng.binomial(1, 0.10, size=n_users)

campaign = pd.DataFrame({
    "user_id": np.arange(1, n_users + 1),
    "variant": variant,
    "opened": opened,
    "clicked": clicked,
    "converted": converted,
})
campaign.head()

## Compare the funnel metrics

These are simulated proportions from one small random draw. Since both variants use the same outcome probabilities, the observed differences are random variation.

In [ ]:
metrics = campaign.groupby("variant")[["opened", "clicked", "converted"]].mean()
metrics.rename(columns={
    "opened": "open_rate",
    "clicked": "click_rate",
    "converted": "conversion_rate",
})

In [ ]:
ax = (metrics * 100).plot(kind="bar", figsize=(8, 4), color=["#176b87", "#ed765a", "#d1a838"])
ax.set_title("Simulated email funnel by variant")
ax.set_ylabel("Rate (%)")
ax.set_xlabel("Subject-line variant")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(["Opened", "Clicked", "Converted"], frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## Test the primary metric

Fisher's exact test compares the simulated open counts in this small sample. The test result describes only this generated dataset; it cannot be used as a campaign conclusion.

In [ ]:
open_table = pd.crosstab(campaign["variant"], campaign["opened"]).reindex(
    index=["A", "B"], columns=[0, 1], fill_value=0
)
open_table.columns = ["not_opened", "opened"]
open_table

In [ ]:
odds_ratio, p_value = fisher_exact(open_table[["not_opened", "opened"]].to_numpy())

print(f"Simulated open rate, A: {metrics.loc['A', 'opened']:.1%}")
print(f"Simulated open rate, B: {metrics.loc['B', 'opened']:.1%}")
print(f"Fisher exact p-value (simulated sample only): {p_value:.3f}")

## Interpretation

The generated sample may show different open rates even though the simulation assigns the same assumed open probability to A and B. That difference is random variation. A non-significant result here does not establish that two real subject lines are equivalent, and a significant result would not establish a real campaign effect.

A real email experiment needs observed campaign data, a pre-defined primary metric, an adequately powered sample size, and a plan for assignment, duration, and stopping. No emails are sent by this notebook.